# Single Foot IMU Stride Analysis

This notebook demonstrates how to estimate walking strides and gait parameters from a single foot-mounted IMU sensor. It loads an APDM `.h5` recording, detects walking bouts, runs inertial mechanization (orientation tracking, footfall detection, zero-velocity updates), and visualizes the resulting stride trajectories and metrics.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremydwong/stride_estimation_imu/blob/main/notebooks/demo_colab_one_foot.ipynb)

In [ ]:
from datetime import time as dt_time

# =============================================================================
# DATA SOURCE: set to 'demo' to use included sample data, or 'upload' to
# upload your own .h5 file in the next cell.
# =============================================================================
DATA_SOURCE = 'demo'  # 'demo' or 'upload'
FOOT_LABEL = 'Left Foot'

# =============================================================================
# WALKING BOUT TARGETS (times in the recording's local timezone)
# Each tuple: (name, target_time, search_window_seconds)
#
# Timezone note: APDM timestamps are UTC microseconds since epoch. The sensor
# config stores the recording timezone, so the module automatically converts
# timestamps to the recording's local time. The target times below should
# match the clock time where/when the data was recorded.
# =============================================================================
WALKING_BOUT_TARGETS = [
    ('track walk', dt_time(16, 34, 0), 240),  # 4:34 PM local recording time
]

# =============================================================================
# BOUT DETECTION PARAMETERS
# =============================================================================
MIN_QUIET_SECONDS = 1.5   # Minimum quiet period to count as walk boundary
MIN_WALK_SECONDS = 3.0    # Minimum walking duration to include
W_THRESHOLD = 30.0        # Max angular velocity (deg/s) for quiet detection
A_THRESHOLD = 1.0         # Max accel deviation from gravity (m/s^2) for quiet

## Setup

Clone the repository and install the required Python packages. If you are running locally (not in Colab), you can skip this cell and install dependencies via `pip install -r requirements.txt`.

In [ ]:
# Clone repository and install dependencies (only needed in Colab)
!git clone https://github.com/jeremydwong/stride_estimation_imu.git 2>/dev/null || true
%cd stride_estimation_imu
!pip install -q numpy scipy matplotlib h5py

In [ ]:
import sys
import os
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

import stride_imu as imu
from stride_imu.apdm import load_imu_recording, ImuRecording
from stride_imu.inertial import detect_walking_bouts, find_bouts_near_time, WalkingBout

print("stride_imu loaded successfully!")

## Load Data

Load an APDM `.h5` sensor file. If `DATA_SOURCE = 'upload'`, this cell lets you upload a file interactively (Colab only). If `DATA_SOURCE = 'demo'`, it uses the included sample data from a foot sensor.

In [ ]:
if DATA_SOURCE == 'upload':
    try:
        from google.colab import files
        print("Select your APDM .h5 file to upload:")
        uploaded = files.upload()
        uploaded_files = sorted(uploaded.keys())
        print(f"\nUploaded {len(uploaded_files)} file(s): {uploaded_files}")
        FOOT_FILE = uploaded_files[0]
        print(f"\nAssigned: FOOT_FILE = {FOOT_FILE}")
        if len(uploaded_files) > 1:
            print(f"Additional files uploaded but not used: {uploaded_files[1:]}")
            print("Re-assign FOOT_FILE manually if needed.")
    except ImportError:
        print("Not running in Colab. Set FOOT_FILE manually.")
        FOOT_FILE = ''
else:
    FOOT_FILE = 'data/20251029-154305_LF_Pilot_Ch_Oct29.h5'
    print(f"Using demo data: FOOT_FILE = {FOOT_FILE}")

## Helper Functions

In [ ]:
class BoutSelection:
    """Stores bout selection with precise timing info."""
    def __init__(self, name, bout, start_datetime, end_datetime, confirmed=False):
        self.name = name
        self.bout = bout
        self.start_datetime = start_datetime
        self.end_datetime = end_datetime
        self.start_idx = bout.start_idx
        self.end_idx = bout.end_idx
        self.duration_seconds = bout.duration_seconds
        self.confirmed = confirmed

    def __repr__(self):
        return (f"BoutSelection('{self.name}', "
                f"start={self.start_datetime.strftime('%H:%M:%S')}, "
                f"duration={self.duration_seconds:.1f}s)")


def compute_alignment_rotation(P):
    """Compute rotation matrix to align trajectory with +Y axis."""
    direction = P[-1, :2] - P[0, :2]
    theta = np.arctan2(direction[1], direction[0])
    rotation_angle = np.pi / 2 - theta
    cos_a, sin_a = np.cos(rotation_angle), np.sin(rotation_angle)
    return np.array([[cos_a, -sin_a, 0], [sin_a, cos_a, 0], [0, 0, 1]])


def apply_rotation(P, R):
    """Apply rotation matrix to trajectory, centered at start."""
    P_centered = P - P[0, :]
    return (R @ P_centered.T).T


def compute_total_distance(P):
    """Compute total path length."""
    diffs = np.diff(P, axis=0)
    return float(np.sum(np.sqrt(np.sum(diffs ** 2, axis=1))))


def compute_step_metrics(strides):
    """Compute step-level metrics from stride segmentation."""
    step_speeds = strides['frwd_speed']
    step_durations = strides['time']
    
    if len(step_speeds) == 0:
        return {'step_lengths': np.array([]), 'step_durations': np.array([]),
                'step_speeds': np.array([]), 'mean_speed': 0.0, 'std_speed': 0.0,
                'mean_length': 0.0, 'std_length': 0.0, 'mean_duration': 0.0,
                'std_duration': 0.0, 'n_steps': 0}
    
    step_lengths = strides['frwd'][-1, :]
    return {
        'step_lengths': step_lengths, 'step_durations': step_durations,
        'step_speeds': step_speeds,
        'mean_speed': np.mean(step_speeds), 'std_speed': np.std(step_speeds),
        'mean_length': np.mean(step_lengths), 'std_length': np.std(step_lengths),
        'mean_duration': np.mean(step_durations), 'std_duration': np.std(step_durations),
        'n_steps': len(step_speeds)
    }


def process_walking_bout(bout, recording, period):
    """Process a single walking bout for one foot."""
    bout_rec = recording[bout.start_idx:bout.end_idx]
    walk_info = imu.compute_position(bout_rec.Wb, bout_rec.Ab, period)
    strides = imu.stride_segmentation(walk_info, period)
    
    return {
        'metrics': compute_step_metrics(strides),
        'strides': strides,
        'walk_info': walk_info,
        'bout_rec': bout_rec,
        'bout': bout,
        'total_distance': compute_total_distance(walk_info['P'])
    }

## Plotting Functions

In [ ]:
def plot_trajectory(walk_info, title=""):
    """Plot foot trajectory aligned to forward direction."""
    P = walk_info['P']
    R = compute_alignment_rotation(P)
    P_rot = apply_rotation(P, R)
    
    fig = plt.figure(figsize=(14, 6))
    fig.suptitle(f'Foot Trajectory - {title}', fontsize=12)
    
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.plot(P_rot[:, 0], P_rot[:, 1], 'b-', linewidth=1, alpha=0.8)
    ax1.plot(P_rot[0, 0], P_rot[0, 1], 'go', markersize=8, label='Start')
    ax1.plot(P_rot[-1, 0], P_rot[-1, 1], 'rs', markersize=8, label='End')
    ax1.set_xlabel('X (lateral) [m]'); ax1.set_ylabel('Y (forward) [m]')
    ax1.set_title('XY View'); ax1.legend(); ax1.grid(True, alpha=0.3); ax1.axis('equal')
    
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    ax2.plot(P_rot[:, 0], P_rot[:, 1], P_rot[:, 2], 'b-', linewidth=1, alpha=0.8)
    ax2.set_xlabel('X [m]'); ax2.set_ylabel('Y [m]'); ax2.set_zlabel('Z [m]')
    ax2.set_title('3D View')
    plt.tight_layout()
    return fig


def plot_bout_summary(bout_name, metrics):
    """Create summary plot for a walking bout."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Walking Bout: {bout_name}', fontsize=14)
    
    # Step speed distribution
    ax = axes[0]
    if metrics['n_steps'] > 0:
        ax.hist(metrics['step_speeds'], bins=20, alpha=0.7, color='blue')
    ax.set_xlabel('Step Speed [m/s]'); ax.set_ylabel('Count')
    ax.set_title('Step Speed Distribution'); ax.grid(True)
    
    # Step length distribution
    ax = axes[1]
    if metrics['n_steps'] > 0:
        ax.hist(metrics['step_lengths'], bins=20, alpha=0.7, color='green')
    ax.set_xlabel('Step Length [m]'); ax.set_ylabel('Count')
    ax.set_title('Step Length Distribution'); ax.grid(True)
    
    # Summary text
    ax = axes[2]
    text = (f"Steps: {metrics['n_steps']}\n\n"
            f"Speed:\n  Mean: {metrics['mean_speed']:.2f} m/s\n"
            f"  Std:  {metrics['std_speed']:.2f} m/s\n\n"
            f"Step Length:\n  Mean: {metrics['mean_length']:.2f} m\n"
            f"  Std:  {metrics['std_length']:.2f} m\n\n"
            f"Step Duration:\n  Mean: {metrics['mean_duration']:.2f} s\n"
            f"  Std:  {metrics['std_duration']:.2f} s")
    ax.text(0.1, 0.5, text, transform=ax.transAxes, fontsize=11,
            verticalalignment='center', fontfamily='monospace')
    ax.axis('off'); ax.set_title('Summary')
    
    plt.tight_layout()
    return fig

## 1. Load IMU Recording

Use `load_imu_recording()` to open the APDM `.h5` file and return an `ImuRecording` containing the gyroscope (`Wb`), accelerometer (`Ab`), magnetometer (`Mb`), timestamps, and sampling period. The sensor orientation transform is applied automatically.

In [ ]:
print("Loading IMU recording...")
foot_rec = load_imu_recording(FOOT_FILE)
PERIOD = foot_rec.period

print(f"  {FOOT_LABEL}: {len(foot_rec)} samples")
print(f"  Sampling period: {PERIOD:.6f} s ({1/PERIOD:.1f} Hz)")
if foot_rec.tz_offset_hours is not None:
    print(f"  Recording timezone: UTC{foot_rec.tz_offset_hours:+.0f}")

## 2. Detect Walking Bouts

Scan the full recording for walking bouts. A walking bout is a period of sustained movement bounded on both sides by quiet (stationary) periods. The algorithm detects quiet periods using angular velocity and acceleration thresholds, then identifies active segments between them that exceed a minimum duration.

In [ ]:
print("Detecting walking bouts...")
all_bouts = detect_walking_bouts(
    foot_rec.Wb, foot_rec.Ab, PERIOD,
    W_threshold=W_THRESHOLD, A_threshold=A_THRESHOLD,
    min_quiet_seconds=MIN_QUIET_SECONDS, min_walk_seconds=MIN_WALK_SECONDS
)

print(f"\nDetected {len(all_bouts)} walking bouts:")
for i, bout in enumerate(all_bouts):
    bout_start = foot_rec.time_datetime[bout.start_idx]
    print(f"  {i+1}. {bout_start.strftime('%H:%M:%S')} - {bout.duration_seconds:.1f}s")

## 3. Select Bouts Near Target Times

Filter the detected bouts to those occurring near the target times specified in the configuration cell. For each target, the longest bout within the search window is selected. This is useful when the recording contains multiple activities and you want to analyze a specific walk.

In [ ]:
bout_selections = []

for bout_name, target_time, search_window in WALKING_BOUT_TARGETS:
    print(f"\nSearching for '{bout_name}' near {target_time.strftime('%H:%M:%S')}...")
    matching = find_bouts_near_time(all_bouts, foot_rec.time_datetime, target_time, search_window)
    
    if matching:
        best = max(matching, key=lambda b: b.duration_seconds)
        start_dt = foot_rec.time_datetime[best.start_idx]
        end_dt = foot_rec.time_datetime[best.end_idx - 1]
        sel = BoutSelection(bout_name, best, start_dt, end_dt, confirmed=True)
        bout_selections.append(sel)
        print(f"  Found: {sel}")
    else:
        print(f"  No bouts found within {search_window}s window")

print(f"\nSelected {len(bout_selections)} bout(s) for analysis")

## 4. Process Walking Bouts

For each selected bout, run the full inertial mechanization pipeline: `compute_position()` integrates angular velocity to track foot orientation, transforms accelerations to the navigation frame, detects footfalls (stance phases), and applies zero-velocity updates to correct drift. Then `stride_segmentation()` extracts individual strides and computes step speed, length, and duration.

In [ ]:
bout_results = {}

for sel in bout_selections:
    print(f"\n{'='*60}")
    print(f"Processing: {sel.name}")
    print(f"{'='*60}")
    
    result = process_walking_bout(sel.bout, foot_rec, PERIOD)
    bout_results[sel.name] = {'selection': sel, **result}
    
    m = result['metrics']
    print(f"{FOOT_LABEL}: {m['n_steps']} steps, {m['mean_speed']:.2f} m/s, {result['total_distance']:.2f} m")

## 5. Visualize Results

For each bout, three plots are generated: (1) stride trajectories showing each step's forward vs. lateral displacement, (2) the full foot trajectory in 2D and 3D after rotation-correcting for walking direction, and (3) a summary panel with histograms of step speed and length distributions along with aggregate statistics.

In [ ]:
for name, r in bout_results.items():
    m = r['metrics']
    
    # Stride trajectories
    if m['n_steps'] > 0:
        fig, ax = plt.subplots(1, 1, figsize=(7, 6))
        imu.plt_ltrl_frwd_strides(r['strides'], show=False)
        ax.set_title(f'{FOOT_LABEL} - {name}')
        plt.tight_layout()
        plt.show()
    
    # Rotation-corrected trajectory
    plot_trajectory(r['walk_info'], title=f'{FOOT_LABEL} - {name}')
    plt.show()
    
    # Bout summary
    plot_bout_summary(name, m)
    plt.show()

## 6. Summary Table

Print a compact table of all analyzed bouts with their start time, duration, number of steps detected, mean walking speed, and total distance walked.

In [ ]:
if bout_results:
    print(f"\n{'Bout':<15} {'Start':<10} {'Duration':>8} {'Steps':>8} {'Speed':>10} {'Distance':>10}")
    print("-" * 65)
    for name, r in bout_results.items():
        sel = r['selection']
        m = r['metrics']
        print(f"{name:<15} {sel.start_datetime.strftime('%H:%M:%S'):<10} "
              f"{sel.duration_seconds:>8.1f} {m['n_steps']:>8} "
              f"{m['mean_speed']:>10.2f} {r['total_distance']:>10.2f}")